In [1]:
import pandas as pd
import json
import numpy as np
import requests
import os

In [2]:
os.getcwd()

'c:\\Users\\Agus\\Bootcamp\\proyectos\\4_PROYECTO_2 EDA\\src\\notebooks'

In [3]:
os.listdir("../data/brutos")


['1_BIS DATA_DATA HUB_PRECIO VIVIENDA.csv',
 '2_OMT_INBOUND TOURISM.xlsx',
 '3_BANCO MUNDIAL_INFLACION POR PAISES.csv',
 '4_EQUIVALENCIAS_PAISES_ISO_3.csv',
 '5_BANCO MUNDIAL_PIB PER CAPITA.csv']

In [4]:
data_vivienda = pd.read_csv("../data/brutos/1_BIS DATA_DATA HUB_PRECIO VIVIENDA.csv")
data_vivienda

,date,country_code,country,price
0,1928-03-31,IT,Italy,-1.7232
1,1928-06-30,IT,Italy,-1.7330
2,1928-09-30,IT,Italy,-2.0023
3,1928-12-31,IT,Italy,-2.5485
4,1929-03-31,IT,Italy,-3.0912
...,...,...,...,...
8822,2026-06-30,SK,Slovakia,13.6164
8823,2026-06-30,TH,Thailand,2.3509
8824,2026-06-30,TR,Türkiye,25.1423
8825,2026-06-30,US,United States,1.1603


In [5]:
#1. Me familiarizo con la estructura del dataset y observo su shape, info, head... 

In [6]:
data_vivienda.info #8827 filas * 4 columnas

<bound method DataFrame.info of             date country_code        country    price
0     1928-03-31           IT          Italy  -1.7232
1     1928-06-30           IT          Italy  -1.7330
2     1928-09-30           IT          Italy  -2.0023
3     1928-12-31           IT          Italy  -2.5485
4     1929-03-31           IT          Italy  -3.0912
...          ...          ...            ...      ...
8822  2026-06-30           SK       Slovakia  13.6164
8823  2026-06-30           TH       Thailand   2.3509
8824  2026-06-30           TR        Türkiye  25.1423
8825  2026-06-30           US  United States   1.1603
8826  2026-06-30           ZA   South Africa   5.8082

[8827 rows x 4 columns]>

In [7]:
data_vivienda.describe()

,price
count,8827.000000
mean,7.002606
std,13.128615
min,-42.251200
25%,1.483900
50%,5.335000
75%,10.349200
max,360.351400


In [8]:
data_vivienda.head(20)

,date,country_code,country,price
0,1928-03-31,IT,Italy,-1.7232
1,1928-06-30,IT,Italy,-1.7330
2,1928-09-30,IT,Italy,-2.0023
3,1928-12-31,IT,Italy,-2.5485
4,1929-03-31,IT,Italy,-3.0912
5,1929-06-30,IT,Italy,-3.6357
6,1929-09-30,IT,Italy,-4.3463
7,1929-12-31,IT,Italy,-5.2430
8,1930-03-31,IT,Italy,-6.1479
9,1930-06-30,IT,Italy,-7.0660


In [9]:
data_vivienda.isnull().sum()

date            0
country_code    0
country         0
price           0
dtype: int64

#2 Agrupo por país (country). Cojo la columna date y, de cada país, la fecha más antigua (min) para ver la serie histórica por países. 

In [10]:
data_vivienda.groupby("country")["date"].min()

country
Advanced economies    2008-12-31
Australia             1971-03-31
Austria               1987-09-30
Belgium               1971-03-31
Brazil                2002-03-31
                         ...    
Thailand              1992-03-31
Türkiye               2011-03-31
United Kingdom        1969-06-30
United States         1971-03-31
World                 2008-12-31
Name: date, Length: 61, dtype: str

#3 Voy a mergearlo con el dataset equivalencias para hacer la correlación país código iso3

In [11]:
equivalencias = pd.read_csv("../data/brutos/4_EQUIVALENCIAS_PAISES_ISO_3.csv")
equivalencias

,codigo_onu,iso2,iso3,nombre_iso
0,4,AF,AFG,Afghanistan
1,8,AL,ALB,Albania
2,10,AQ,ATA,Antarctica
3,12,DZ,DZA,Algeria
4,16,AS,ASM,American Samoa
...,...,...,...,...
244,862,VE,VEN,"Venezuela, Bolivarian Republic of"
245,876,WF,WLF,Wallis and Futuna
246,882,WS,WSM,Samoa
247,887,YE,YEM,Yemen


In [12]:
data_vivienda = data_vivienda.merge(equivalencias[["iso2", "iso3"]], left_on="country_code", right_on="iso2", how="left")
data_vivienda


,date,country_code,country,price,iso2,iso3
0,1928-03-31,IT,Italy,-1.7232,IT,ITA
1,1928-06-30,IT,Italy,-1.7330,IT,ITA
2,1928-09-30,IT,Italy,-2.0023,IT,ITA
3,1928-12-31,IT,Italy,-2.5485,IT,ITA
4,1929-03-31,IT,Italy,-3.0912,IT,ITA
...,...,...,...,...,...,...
8822,2026-06-30,SK,Slovakia,13.6164,SK,SVK
8823,2026-06-30,TH,Thailand,2.3509,TH,THA
8824,2026-06-30,TR,Türkiye,25.1423,TR,TUR
8825,2026-06-30,US,United States,1.1603,US,USA


#4 le doy la estructura casi final en cuanto al shape de la tabla

In [13]:
data_vivienda = data_vivienda[["country", "iso3", "date", "price"]]
data_vivienda

,country,iso3,date,price
0,Italy,ITA,1928-03-31,-1.7232
1,Italy,ITA,1928-06-30,-1.7330
2,Italy,ITA,1928-09-30,-2.0023
3,Italy,ITA,1928-12-31,-2.5485
4,Italy,ITA,1929-03-31,-3.0912
...,...,...,...,...
8822,Slovakia,SVK,2026-06-30,13.6164
8823,Thailand,THA,2026-06-30,2.3509
8824,Türkiye,TUR,2026-06-30,25.1423
8825,United States,USA,2026-06-30,1.1603


#5 Homogeneizo nombre de las columnas usando un diccionario

In [14]:
data_vivienda = data_vivienda.rename(columns={
    "country": "pais",
    "price": "crec._precio_(%)",
    "date": "anio"})
data_vivienda

,pais,iso3,anio,crec._precio_(%)
0,Italy,ITA,1928-03-31,-1.7232
1,Italy,ITA,1928-06-30,-1.7330
2,Italy,ITA,1928-09-30,-2.0023
3,Italy,ITA,1928-12-31,-2.5485
4,Italy,ITA,1929-03-31,-3.0912
...,...,...,...,...
8822,Slovakia,SVK,2026-06-30,13.6164
8823,Thailand,THA,2026-06-30,2.3509
8824,Türkiye,TUR,2026-06-30,25.1423
8825,United States,USA,2026-06-30,1.1603


#6 Paso de string a int64 los años. Me quedo con los 4 primoros caracteres (año)

In [15]:
data_vivienda["anio"] = data_vivienda["anio"].str[:4].astype(int)
print(data_vivienda["anio"].dtype)
data_vivienda.head()

int64


,pais,iso3,anio,crec._precio_(%)
0,Italy,ITA,1928,-1.7232
1,Italy,ITA,1928,-1.7330
2,Italy,ITA,1928,-2.0023
3,Italy,ITA,1928,-2.5485
4,Italy,ITA,1929,-3.0912


#7 Me quedo con la serie histórica de 2015 a 2024, para homogeneizar con dataset de OMT y resto de dataset

In [16]:
data_vivienda = data_vivienda[data_vivienda["anio"].between(2015, 2024)]
data_vivienda = data_vivienda.reset_index(drop=True)
print(data_vivienda.shape)
print(data_vivienda["anio"].min(), data_vivienda["anio"].max())
print(data_vivienda)

(2440, 4)
2015 2024
                                       pais iso3  anio  crec._precio_(%)
0     Emerging market economies (aggregate)  NaN  2015            3.1486
1                        Advanced economies  NaN  2015            3.6334
2                                   Austria  AUT  2015            3.5670
3                                 Australia  AUS  2015            6.9177
4                                   Belgium  BEL  2015            0.5235
...                                     ...  ...   ...               ...
2435                                Türkiye  TUR  2024           28.5446
2436                          United States  USA  2024            3.4585
2437                              Euro area  NaN  2024            4.3084
2438                                  World  NaN  2024            2.0937
2439                           South Africa  ZAF  2024            1.0748

[2440 rows x 4 columns]


#8 Limpio los NaN de iso 3 con subset iso 3

In [17]:
data_vivienda = data_vivienda.dropna(subset=["iso3"])
print(data_vivienda.shape)

(2280, 4)


In [18]:
print(data_vivienda.groupby(["iso3", "anio"]).size().max())

4


In [ ]:
#9 Hago la media anual (Cada año contiene fluctuación del precio de la vivienda por trimestre y hay que simpificar la data para hacer merge conjunto), pero primero verifico que efectivamente tengo 4Quatrimestres. Redondeo a un decimal y reset index. 

In [19]:
print(data_vivienda.groupby(["iso3", "anio"]).size().max())

4


In [21]:
data_vivienda = data_vivienda.groupby(["pais", "iso3", "anio"])["crec._precio_(%)"].mean()

In [23]:
data_vivienda = data_vivienda.round(1)

In [24]:
data_vivienda = data_vivienda.reset_index()

In [25]:
print(data_vivienda)

              pais iso3  anio  crec._precio_(%)
0        Australia  AUS  2015               9.0
1        Australia  AUS  2016               5.5
2        Australia  AUS  2017               8.4
3        Australia  AUS  2018              -1.4
4        Australia  AUS  2019              -4.0
..             ...  ...   ...               ...
565  United States  USA  2020               6.7
566  United States  USA  2021              15.1
567  United States  USA  2022              12.8
568  United States  USA  2023               4.0
569  United States  USA  2024               4.3

[570 rows x 4 columns]


# 10 Hago verificaciones finales

print(data_turistas_pernoctan.shape)
print(data_turistas_pernoctan["iso3"].nunique())
print(data_turistas_pernoctan.groupby(["iso3", "anio"]).size().max())
print(data_turistas_pernoctan["crec._turistas_(%)"].isnull().sum())

# 11. Guardo el dataset limpio

In [26]:
data_vivienda.to_csv("../data/depurados/limpio_vivienda.csv", index=False)